In [ ]:
### Installing necessaary packages
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm

# Setup device (GPU if available, else CPU)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
### Mounting Drive to read the label file and image dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_path = '/content/drive/MyDrive/Case Studies/1. Toxic Comment Classification/train.csv'
test_path = '/content/drive/MyDrive/Case Studies/1. Toxic Comment Classification/test.csv'
test_lbl_path = '/content/drive/MyDrive/Case Studies/1. Toxic Comment Classification/test_labels.csv'

In [ ]:
### Importing datasets
import pandas as pd
df = pd.read_csv(train_path)
df_test = pd.read_csv(test_path,encoding='latin')
df_labels = pd.read_csv(test_lbl_path)

In [ ]:
df.head()

In [ ]:
df_test.head()

In [ ]:
df.columns

In [ ]:
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Sum across the columns for each row
sumrow = df[labels].sum(axis=1)

# Count where sum is 0
clean_row = (sumrow == 0).sum()

# Count where sum is > 0
toxic_row = (sumrow > 0).sum()

print("Clean rows:", clean_row)
print("Toxic rows:", toxic_row)

In [ ]:
### Cleaning train dataset
import re

def fast_clean(text):
    # 1. Remove URLs first (so BS doesn't get confused)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 2. Remove HTML tags using Regex (much faster than BeautifulSoup)
    text = re.sub(r'<.*?>', '', text)

    # 3. Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7f]', r'', text)

    # 4. Cleanup whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply this to your dataframe
df['comment_text_cleaned'] = df['comment_text'].apply(fast_clean)

In [ ]:
### Tokenizer Class
class ToxicityDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.text = df['comment_text'].tolist()
        self.labels = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].values
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])

    # Tokenizer usage
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
            )
        return {
          'input_ids': inputs['input_ids'].squeeze(0),
          'attention_mask': inputs['attention_mask'].squeeze(0),
          'labels': torch.tensor(self.labels[index], dtype=torch.float)
           }

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=6
)
model.to(device)

from torch.utils.data import random_split

dataset = ToxicityDataset(df, tokenizer)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
from torch.optim import AdamW

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3

# Mixed precision scaler
scaler = GradScaler()

for epoch in range(epochs):
    model.train()
    train_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device)

        # Mixed precision forward
        with autocast():
            outputs = model(input_ids, attention_mask=attention_mask, labels=targets)
            loss = outputs.loss

        # Backprop with scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

In [ ]:
train_loss /= len(train_loader)
# Validation
model.eval()
val_loss = 0
with torch.no_grad():
    loop_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]  ", leave=False)
    for batch in loop_val:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device)

        with autocast():
            outputs = model(input_ids, attention_mask=attention_mask, labels=targets)
            val_loss += outputs.loss.item()
        loop_val.set_postfix(val_loss=outputs.loss.item())

val_loss /= len(val_loader)
print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")



In [ ]:
df_test_full = pd.merge(df_test, df_labels, on='id')
df_test_full = df_test_full[df_test_full['toxic'] != -1].reset_index(drop=True)

# Define test Dataset Class so that it matches the training setup
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

target_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

test_dataset = ToxicDataset(
    texts=df_test_full.comment_text.to_numpy(),
    labels=df_test_full[target_columns].to_numpy(),
    tokenizer=tokenizer # Use the same tokenizer as training
)

test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Switch model to evaluation mode
model.eval()

all_targets = []
all_preds = []
all_probs=[]

# No gradients needed during evaluation
with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device)

        # Get model outputs
        outputs = model(input_ids, attention_mask=mask)
        logits = outputs.logits

        # Apply sigmoid since it's multi-label
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()  # threshold of 0.5

        all_targets.append(targets.cpu())
        all_preds.append(preds.cpu())
        all_probs.append(probs.cpu().numpy())

# Combine batches
#all_probs = all_probs.append(probs.cpu().numpy())
all_targets = torch.cat(all_targets, dim=0).numpy()
all_preds = torch.cat(all_preds, dim=0).numpy()

# Compute evaluation metrics
for i, col in enumerate(target_columns):
    auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
    f1 = f1_score(all_targets[:, i], all_preds[:, i])
    print(f"{col}: AUC={auc:.4f}, F1={f1:.4f}")


In [ ]:
# Combine all batches into one array
import numpy as np
final_probs = np.vstack(all_probs)

In [ ]:
final_probs

In [ ]:
# Create the DataFrame
submission_df = pd.DataFrame(final_probs, columns=target_columns)
# Insert the ID column at the start
submission_df.insert(0, 'id', df_test_full['id'])
# Save to CSV
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")